# LOOK: UK Biobank All-Evidence Glaucoma Benchmark

Participant-level bilateral CFP and central 2D OCT classification with MHD V4. The selected task uses 925 all-evidence record-derived glaucoma cases and 925 matched controls. The task scout is complete; formal baseline validation is now the default, and sealed tests require frozen manifests.


## Environment and project

Select the `LOOK` kernel registered by Step 3. The package must already be installed; the notebook never modifies `sys.path`.


In [ ]:
from pathlib import Path
import json
import os

print("Select the LOOK kernel before running all cells.")


## Single configuration cell

Use singleton lists for one experiment or add values to expand a grid. `baseline_selection` executes the preregistered calibration, unimodal references, seven fusion positions, and Top-3 confirmation. `custom_validation` executes exactly the lists below.


In [ ]:
PROJECT_ROOT = Path("/home/mengh/LOOK/2026_09_03_19_35_04")
EXECUTION_MODE = "baseline_selection"  # baseline_selection | custom_validation | task_scout | freeze | test
EXECUTE = True
GPU_DEVICES = [0, 1]  # [0] for one GPU; [0, 1] for DDP
CHECK_ALL_IMAGE_PATHS = True
BOOTSTRAP_ITERATIONS = 2000
TASK_SCOUT_PROFILES = [
    "glaucoma_high_confidence", "glaucoma_all_evidence",
    "any_target_eye_disease_high_confidence", "any_target_eye_disease",
    "glaucoma_objective_only",
    "diabetic_eye_disease_all_evidence",
    "macular_degeneration_all_evidence",
]
TASK_SCOUT_FUSION_POSITIONS = ["feature"]
TASK_SCOUT_EPOCHS = 12
TASK_SCOUT_PATIENCE = 4

FUSION_POSITIONS = ["feature"]  # input, stem, layer1, layer2, layer3, layer4, feature
SEEDS = [3407]
FILLING_STRATEGIES = ["raw_zero"]  # raw_zero, normalized_mean, paired_cgan

CLASSIFIER_PROFILES = [{
    "name": "custom",
    "epochs": 100, "patience": 15,
    "effective_batch_size": 128, "micro_batch_size": 64, "num_workers": 8,
    "pretrained_lr": 1e-4, "new_layer_lr": 1e-3,
    "weight_decay": 1e-4, "warmup_epochs": 5,
    "sampling_strategy": "natural_without_replacement",
    "loss_name": "cross_entropy", "label_smoothing": 0.0,
    "classifier_dropout": 0.2, "training_strategy": "end_to_end_finetuning",
    "amp": True, "baseline_auroc_target": 0.80,
    "baseline_macro_f1_target": 0.70,
    "baseline_min_sensitivity": 0.65, "baseline_min_specificity": 0.65,
}]
GAN_PROFILES = [{
    "name": "primary", "gan_validation_fraction": 0.1,
    "gan_epochs": 100, "gan_patience": 10,
    "gan_effective_batch_size": 448, "gan_batch_size": 112,
    "gan_num_workers": 8, "gan_learning_rate": 2e-4,
    "gan_beta1": 0.5, "gan_lambda_l1": 100.0, "gan_base_channels": 64,
}]
LOOK_PROFILES = [{
    "name": "primary", "enabled": True,
    "evaluate_random_missing": True, "evaluate_missing_baselines": True,
    "missing_patterns": ["oct_missing", "cfp_missing"],
    "missing_ratios": [0.2, 0.4, 0.6, 0.8, 1.0],
    "correction_nodes": ["all_available"],
    "downsample_factors": [4, 8, 16],
    # PCA Dmax is independent of the validation search candidates.
    # Alternatively use list(range(32, 513, 32)) for a regular search step.
    "latent_dims": [8, 16, 32, 64, 96, 128, 192, 256, 384, 512],
    "max_pca_rank": 512,  # Shared complete-training PCs, not missing-specific.
    "primary_metric": "macro_auroc_ovr",
}]
BASELINE_SELECTION_MANIFEST = None
FROZEN_STUDY_MANIFEST = None


## Resolve paths and study grid

Paths come from explicit configuration, environment variables, then `project.json`. Existing valid artifacts are resumed by deterministic run ID.


In [ ]:
import torch
from look_core.paths import ProjectPaths
from look_core.study_grid import StudyGrid, expand_study_grid
from look_core.task_selection import validate_selected_task

assert (PROJECT_ROOT / "project.json").is_file(), PROJECT_ROOT
os.environ["LOOK_PROJECT_ROOT"] = str(PROJECT_ROOT)
project = json.loads((PROJECT_ROOT / "project.json").read_text())
paths = ProjectPaths.load(project_root=PROJECT_ROOT)
assert paths.labels_csv.is_file(), "Run Steps 21-22 before experiments"
selected_task = validate_selected_task(paths)
grid = StudyGrid(
    fusion_positions=FUSION_POSITIONS, seeds=SEEDS,
    filling_strategies=FILLING_STRATEGIES,
    classifier_profiles=CLASSIFIER_PROFILES,
    gan_profiles=GAN_PROFILES, look_profiles=LOOK_PROFILES,
)
cases = expand_study_grid(
    grid, paths, phase="validation", gpu_devices=tuple(GPU_DEVICES),
    check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
print({"mode": EXECUTION_MODE, "execute": EXECUTE, "task": selected_task["selected_task_profile"], "cases": len(cases), "runs": str(paths.runs_root)})


## Cohort audit

This cell verifies class mapping, participant uniqueness, split counts, and referenced paths before any formal run.


In [ ]:
from look_core.data import validate_reference_table

audit = validate_reference_table(paths.labels_csv, paths.image_root, check_paths=CHECK_ALL_IMAGE_PATHS)
print(json.dumps(audit, indent=2))


## Execute or resume

Results, checkpoints, predictions, monitor curves, and progress JSON are written under the new timestamp's `runs/` tree. Test access is impossible without a frozen manifest.


In [ ]:
from look_core.study_grid import (
    freeze_study_grid, run_baseline_selection, run_study_grid,
    study_grid_from_baseline_selection,
)
from look_core.task_scout import run_task_scout

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
gpus = tuple(GPU_DEVICES)
if EXECUTION_MODE == "task_scout":
    result = run_task_scout(
        paths, profile_ids=TASK_SCOUT_PROFILES,
        fusion_positions=TASK_SCOUT_FUSION_POSITIONS, seed=SEEDS[0],
        epochs=TASK_SCOUT_EPOCHS, patience=TASK_SCOUT_PATIENCE,
        gpu_devices=gpus, execute=EXECUTE,
    )
elif EXECUTION_MODE == "baseline_selection":
    result = run_baseline_selection(
        paths, device, execute=EXECUTE,
        check_all_image_paths=CHECK_ALL_IMAGE_PATHS, gpu_devices=gpus,
    )
elif EXECUTION_MODE == "custom_validation":
    result = run_study_grid(
        grid, paths, device, execute=EXECUTE, phase="validation",
        check_all_image_paths=CHECK_ALL_IMAGE_PATHS,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS, gpu_devices=gpus,
    )
elif EXECUTION_MODE == "freeze":
    if not BASELINE_SELECTION_MANIFEST:
        raise ValueError("BASELINE_SELECTION_MANIFEST is required")
    frozen_grid = study_grid_from_baseline_selection(Path(BASELINE_SELECTION_MANIFEST))
    result = freeze_study_grid(frozen_grid, paths, device, gpu_devices=gpus)
elif EXECUTION_MODE == "test":
    if not BASELINE_SELECTION_MANIFEST or not FROZEN_STUDY_MANIFEST:
        raise ValueError("Both baseline and frozen study manifests are required")
    frozen_grid = study_grid_from_baseline_selection(Path(BASELINE_SELECTION_MANIFEST))
    result = run_study_grid(
        frozen_grid, paths, device, execute=EXECUTE, phase="test",
        frozen_manifest=Path(FROZEN_STUDY_MANIFEST),
        bootstrap_iterations=BOOTSTRAP_ITERATIONS, gpu_devices=gpus,
    )
else:
    raise ValueError(EXECUTION_MODE)
print(json.dumps(result, indent=2, default=str))


## Inspect durable progress

The newest progress and leaderboard files can be read while detached training continues.


In [ ]:
progress = sorted([*paths.runs_root.glob("task_scout/*/progress.json"), *paths.runs_root.glob("sweeps/*/progress.json")], key=lambda p: p.stat().st_mtime)
leaderboards = sorted([*paths.runs_root.glob("task_scout/*/leaderboard.csv"), *paths.runs_root.glob("sweeps/*/leaderboard.csv")], key=lambda p: p.stat().st_mtime)
print("Latest progress:", progress[-1] if progress else None)
print("Latest leaderboard:", leaderboards[-1] if leaderboards else None)
if progress:
    print(progress[-1].read_text())
